# Topic Modelling для BBC-новостей

In [1]:
from pathlib import Path
import re
import numpy as np
import pandas as pd

import matplotlib.pyplot as plt
import seaborn as sns

from gensim.corpora import Dictionary
from gensim.models import TfidfModel
from gensim.models import ldamodel
from gensim.parsing.preprocessing import STOPWORDS

RANDOM_STATE = 42

### 1. Загрузка данных

In [2]:
def load_bbc_dataset(data_dir: Path):
    rows = []
    for class_dir in sorted(data_dir.iterdir()):
        if not class_dir.is_dir():
            continue
        label = class_dir.name
        for path in sorted(class_dir.glob('*.txt')):
            text = path.read_text(encoding='latin-1', errors='ignore')
            text = re.sub(r'\s+', ' ', text).strip()
            rows.append({'text': text, 'label': label, 'path': str(path)})
    return pd.DataFrame(rows)


df = load_bbc_dataset(Path('data/bbc'))
print(df.shape)
print(df['label'].value_counts())
df.head()

(2221, 3)
label
sport            511
business         506
politics         417
tech             401
entertainment    386
Name: count, dtype: int64


,text,label,path
0,Ad sales boost Time Warner profit Quarterly pr...,business,data/bbc/business/001.txt
1,Dollar gains on Greenspan speech The dollar ha...,business,data/bbc/business/002.txt
2,Yukos unit buyer faces loan claim The owners o...,business,data/bbc/business/003.txt
3,High fuel prices hit BA's profits British Airw...,business,data/bbc/business/004.txt
4,Pernod takeover talk lifts Domecq Shares in UK...,business,data/bbc/business/005.txt


### 2. Предобработка текста

In [3]:
def preprocess_for_gensim(text: str) -> list[str]:
    # lowercase, очистка от не-буквенных символов, stopwords, токены.
    text = text.lower()
    text = re.sub(r'[^a-z\s]', ' ', text)
    tokens = [token for token in text.split() if token not in STOPWORDS and len(token) > 2]
    return tokens


df['tokens'] = df['text'].apply(preprocess_for_gensim)
df['clean_text'] = df['tokens'].apply(lambda tokens: ' '.join(tokens))
df[['label', 'tokens']].head()

,label,tokens
0,business,"[sales, boost, time, warner, profit, quarterly..."
1,business,"[dollar, gains, greenspan, speech, dollar, hit..."
2,business,"[yukos, unit, buyer, faces, loan, claim, owner..."
3,business,"[high, fuel, prices, hit, profits, british, ai..."
4,business,"[pernod, takeover, talk, lifts, domecq, shares..."


### 3. Topic modeling используя LDA

In [4]:
def train_gensim_lda_for_class(tokens_by_doc, n_topics=4, n_top_words=12):
    dictionary = Dictionary(tokens_by_doc)
    # убираем слова которые встречаются меньше чем в 3 документах и больше чем в 85% документов для снижения размерности и шума
    dictionary.filter_extremes(no_below=3, no_above=0.85)

    corpus = [dictionary.doc2bow(tokens) for tokens in tokens_by_doc]
    tfidf = TfidfModel(corpus)
    corpus_tfidf = list(tfidf[corpus])

    lda = ldamodel.LdaModel(
        corpus=corpus,
        id2word=dictionary,
        num_topics=n_topics,
        alpha='auto',
        eta='auto',
        iterations=50,
        passes=8,
        random_state=RANDOM_STATE,
    )

    doc_topic = np.zeros((len(corpus), n_topics), dtype=float)
    for doc_idx, bow in enumerate(corpus):
        for topic_id, weight in lda.get_document_topics(bow, minimum_probability=0):
            doc_topic[doc_idx, topic_id] = weight

    topic_words = []
    for topic_id in range(n_topics):
        words = lda.show_topic(topic_id, topn=n_top_words)
        topic_words.append({
            'topic': topic_id,
            'words': [word for word, _ in words],
            'weights': [float(weight) for _, weight in words],
        })

    return {
        'dictionary': dictionary,
        'corpus': corpus,
        'tfidf': tfidf,
        'corpus_tfidf': corpus_tfidf,
        'lda': lda,
        'doc_topic': doc_topic,
        'topic_words': topic_words,
    }


def print_topic_words(topic_words):
    for item in topic_words:
        words_with_weights = [
            f"{word} ({weight:.3f})"
            for word, weight in zip(item['words'], item['weights'])
        ]
        print(f"Topic {item['topic']}:", ', '.join(words_with_weights))

In [5]:
N_TOPICS = 4
N_TOP_WORDS = 12

lda_results = {}

for label in sorted(df['label'].unique()):
    tokens_by_doc = df.loc[df['label'] == label, 'tokens'].tolist()
    result = train_gensim_lda_for_class(
        tokens_by_doc,
        n_topics=N_TOPICS,
        n_top_words=N_TOP_WORDS,
    )
    lda_results[label] = result

    print('\n' + '=' * 80)
    print(label.upper())
    print_topic_words(result['topic_words'])


BUSINESS
Topic 0: economy (0.011), growth (0.010), year (0.010), economic (0.010), dollar (0.006), government (0.006), market (0.005), world (0.005), china (0.005), spending (0.004), budget (0.004), trade (0.004)
Topic 1: shares (0.008), market (0.007), company (0.007), firm (0.007), bid (0.007), new (0.006), offer (0.006), oil (0.006), year (0.006), deal (0.006), deutsche (0.005), firms (0.005)
Topic 2: company (0.010), yukos (0.009), firm (0.009), government (0.005), russian (0.005), new (0.005), companies (0.005), court (0.005), business (0.005), group (0.005), state (0.005), deal (0.005)
Topic 3: year (0.015), sales (0.013), market (0.008), prices (0.008), new (0.007), rise (0.006), growth (0.006), bank (0.006), december (0.005), rate (0.005), rates (0.005), figures (0.005)

ENTERTAINMENT
Topic 0: music (0.020), said (0.016), new (0.009), band (0.007), year (0.007), chart (0.006), album (0.005), people (0.005), time (0.004), sales (0.004), record (0.004), industry (0.004)
Topic 1:

Вероятность рядом со словом показывает вес слова внутри конкретного LDA-топика: чем выше значение, тем сильнее слово характеризует этот топик.

### 4. pyLDAvis

In [6]:
import pyLDAvis
import pyLDAvis.gensim_models as gensimvis

label_for_vis = 'business'
result = lda_results[label_for_vis]
vis_data = gensimvis.prepare(result['lda'], result['corpus'], result['dictionary'])
pyLDAvis.display(vis_data)

### 5. Интерпретация топиков по классам

### Business

В классе `business` основные темы связаны с экономикой, рынками и деятельностью компаний. По словам `economy`, `growth`, `economic`, `dollar`, `trade` видно, что часть новостей описывает макроэкономику, валюты, торговлю и экономический рост. Слова `shares`, `market`, `company`, `firm`, `bid`, `offer`, `deal` указывают на фондовый рынок, корпоративные сделки и поглощения. Также встречаются темы продаж, цен, банков и процентных ставок (`sales`, `prices`, `bank`, `rate`, `rates`). В целом этот класс описывает финансово-экономическую и корпоративную повестку.

### Entertainment

В классе `entertainment` топики в основном группируются вокруг музыки, кино и премий. Слова `music`, `band`, `chart`, `album`, `record`, `industry` показывают музыкальную индустрию: альбомы, чарты, группы и продажи. Другая заметная часть корпуса относится к кино: `film`, `movie`, `actor`, `actress`, `star`, `director`, `box office`. Отдельно выделяется наградная повестка через слова `awards`, `award`, `best`, `won`, `oscar`. В целом класс отражает новости культуры, киноиндустрии, музыки, знаменитостей и развлекательных событий.

### Politics

Класс `politics` описывает государственную политику, партии, выборы и общественные реформы. Частые слова `government`, `minister`, `public`, `plans`, `law`, `rights`, `police` указывают на решения правительства, законодательство и работу государственных институтов. Слова `labour`, `election`, `party`, `blair`, `brown`, `howard`, `tax` показывают партийную борьбу, выборы и налоговую повестку. Также видны социально-политические темы вроде `asylum`, `children`, `cards`, `health`. В целом класс посвящен внутренней политике Великобритании, выборам, партиям и государственным решениям.

### Sport

В классе `sport` основные темы связаны с матчами, командами, турнирами и результатами соревнований. По словам `club`, `liverpool`, `chelsea`, `league`, `players`, `team`, `cup` хорошо видна футбольная тематика. Слова `england`, `wales`, `ireland`, `match`, `game`, `win`, `half` отражают матчи сборных и командные виды спорта. Также встречаются международные соревнования и индивидуальные достижения: `world`, `open`, `champion`, `olympic`, `final`, `athens`. В целом класс описывает спортивные события, футбольные клубы, сборные, турниры, победы и травмы игроков.

### Tech

Класс `tech` охватывает интернет, программное обеспечение, игры, мобильные технологии и цифровой контент. Слова `online`, `broadband`, `net`, `internet`, `web`, `users` указывают на интернет-сервисы и инфраструктуру. Слова `software`, `microsoft`, `security`, `search`, `data` относятся к ПО, безопасности, поиску и технологическим компаниям. Отдельный пласт связан с играми (`game`, `games`) и мобильными устройствами/контентом (`mobile`, `phone`, `digital`, `video`, `music`, `sony`, `content`). В целом этот класс описывает цифровые технологии, интернет, софт, мобильные устройства, игры и медиа-контент.